# 02. Multi-Agent Handoffs

This notebook shows specialization and delegation. One agent cleans OCR, another extracts metadata, and a third writes a short summary. The SDK README presents this as a multi-agent workflow: agents can hand off to each other or be exposed as tools so a coordinator can delegate specific subproblems.

## Why this pattern matters
For beginners, the big intuition is that one model does not need to do everything. In humanities workflows, different stages often require different behaviors: careful transcription, factual extraction, and interpretive writing. Separating those responsibilities makes the pipeline easier to debug and easier to explain to students.

This also helps students see that orchestration is not magic. It is a sequence of explicit roles with clear inputs and outputs.

## Concepts
- Agents as tools
- Handoffs
- Pipeline design
- Separation of concerns
- Traceable intermediate outputs

The official docs on [Agents as tools](https://openai.github.io/openai-agents-python/tools/#agents-as-tools) and [Handoffs](https://openai.github.io/openai-agents-python/handoffs/) are especially relevant here.

## Tracing
View traces in the OpenAI Traces dashboard: https://platform.openai.com/traces

## Dataset
A noisy OCR fragment is included in `notebooks/common.py`.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd().parent / '.env')
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env or your environment before running this notebook.'

from agents import set_tracing_export_api_key
set_tracing_export_api_key(os.environ['OPENAI_API_KEY'])

from common import LETTER_TEXTS
from agents import Agent, Runner

raw_text = LETTER_TEXTS[2]['text']
print(raw_text)

## Step 1: Define the specialist agents

The cleaner focuses on transcription quality. The extractor focuses on structured facts. The summarizer focuses on prose.

The key teaching point is that each agent receives a narrower instructions block. Narrower agents are easier to test because a failure usually means a single stage needs adjustment.

In [ ]:
cleaner = Agent(
    name='OCR Cleaner',
    instructions=(
        'Fix obvious OCR mistakes while preserving original meaning. '
        'Do not invent new information.'
    ),
)

extractor = Agent(
    name='Metadata Extractor',
    instructions='Extract people, places, dates, and key facts from cleaned historical text.',
)

summarizer = Agent(
    name='Historical Summarizer',
    instructions='Write a two-sentence summary grounded in the provided cleaned text and extracted facts.',
)

cleaner, extractor, summarizer

## Step 2: Use agents as tools

The coordinator can call other agents with `as_tool`. That is often the easiest way to teach agents-as-tools because it feels like ordinary function calling, but the function happens to be another agent.

Conceptually, this is useful when you want one agent to remain in charge while delegating a subtask to a specialist. The coordinator does not need to know the specialist's internals, only its purpose.

In [ ]:
cleaner_tool = cleaner.as_tool('clean_ocr', 'Clean OCR text while preserving meaning.')
extractor_tool = extractor.as_tool('extract_metadata', 'Extract structured metadata from cleaned text.')
summarizer_tool = summarizer.as_tool('summarize_history', 'Write a short summary based on cleaned text and facts.')

print(cleaner_tool.name, extractor_tool.name, summarizer_tool.name)

## Step 3: Coordinator agent

The coordinator orchestrates the work. This gives students a concrete view of delegation without hiding the steps.

In more advanced workflows, the coordinator can also be taught to stop early, branch to a different specialist, or ask for human review. That is the bridge from simple handoffs to full agentic pipelines.

In [ ]:
coordinator = Agent(
    name='Pipeline Coordinator',
    instructions=(
        'First clean OCR, then extract metadata, then write a short summary. '
        'Always prefer evidence over guessing.'
    ),
    tools=[cleaner_tool, extractor_tool, summarizer_tool],
)

coordinator

## Step 4: Run the pipeline

This call is fully runnable once the API key is set. In notebooks, use `await Runner.run(...)` rather than `run_sync(...)` because the kernel already manages an event loop. Students can inspect the result and compare the intermediate outputs conceptually.

If you want to connect this to the SDK docs, this is the place to introduce tracing and the idea that each delegation can be followed in the run history.

## Further reading
- Agents as tools: https://openai.github.io/openai-agents-python/tools/#agents-as-tools
- Handoffs: https://openai.github.io/openai-agents-python/handoffs/
- Sessions: https://openai.github.io/openai-agents-python/sessions/


In [ ]:
from agents import trace

with trace('DH multi-agent pipeline'):
    result = await Runner.run(coordinator, raw_text)
print(result.final_output)


## Exercise

Add a fourth agent that classifies whether the text looks like a letter, notice, or newspaper excerpt.

## Solution

Create `classifier = Agent(...)`, wrap it with `classifier.as_tool(...)`, add it to `coordinator.tools`, and update the instructions to call it before summarization.